# Plugin Development Tutorial

The EWIS plugin system lets teams extend the toolkit without modifying core code.
Plugins are self-contained units of analysis that hook into the engine pipeline,
receive a shared payload, and return structured results. This architecture keeps
the core deterministic while allowing domain teams to contribute specialized
analytics — from PUE tracking to cooling optimization — as first-class components.

This tutorial walks through building a custom **PUE Efficiency Analyzer** plugin
from scratch. You will learn how to define, test, register, and compose plugins,
as well as how to leverage the plugin context for caching and configuration. By
the end, you will have a fully working plugin that classifies data-center power
usage effectiveness into efficiency tiers.

## Plugin Architecture Overview

Every EWIS plugin extends **`BasePlugin`** and follows a three-method lifecycle:

- **`initialize(context)`** — Called once when the plugin is registered with the
  engine. Use this to validate configuration, pre-load lookup tables, or set up
  connections.
- **`execute(payload, context)`** — Called on every engine run. Receives the
  shared payload dictionary and must return a `PluginResult`.
- **`teardown(context)`** — Called when the engine shuts down. Override this to
  release resources (file handles, network connections, etc.).

Plugins communicate exclusively through two data structures:

- **`PluginResult`** — A frozen dataclass with four fields:
  - `name` (str) — identifies which plugin produced the result
  - `ok` (bool) — `True` if the plugin ran successfully
  - `data` (dict) — the plugin's output values
  - `metadata` (dict) — optional provenance info (timestamps, versions, etc.)

- **`PluginContext`** — Provided by the engine and shared across plugins:
  - `config` (dict) — engine-level configuration
  - `allow_side_effects` (bool) — whether I/O is permitted
  - `plugin_timeout_s` (int) — maximum execution time
  - `cache` (dict) — mutable dict that persists across calls within a session
  - `runtime_metadata` (dict) — read-only run metadata
  - `get(key, default)` — safe accessor for config values

## Step 1 — Define the Plugin Class

We will build a `PUEEfficiencyPlugin` that analyzes **Power Usage Effectiveness**
(PUE). PUE is the ratio of total facility power to IT equipment power — a value
of 1.0 means every watt goes to compute; anything above that represents overhead
(cooling, lighting, power distribution losses).

Our plugin will:
1. Extract PUE, total power, and IT load from the payload.
2. Calculate overhead and efficiency percentage.
3. Classify the result into one of four tiers: *excellent*, *good*, *average*,
   or *poor*.

In [ ]:
from ewis.core.plugin_manager import BasePlugin, PluginResult
from ewis.core.context import PluginContext


class PUEEfficiencyPlugin(BasePlugin):
    """Analyze PUE and classify into efficiency tiers."""

    def __init__(self, name: str = "pue_efficiency") -> None:
        super().__init__(name)

    def initialize(self, context: PluginContext) -> None:
        # No initialization state needed for this plugin.
        pass

    def execute(self, payload: dict, context: PluginContext) -> PluginResult:
        pue = payload["pue"]
        power_mw = payload["power_mw"]
        it_load_mw = payload["it_load_mw"]

        overhead_mw = power_mw - it_load_mw
        efficiency_pct = (it_load_mw / power_mw) * 100 if power_mw > 0 else 0.0

        if pue < 1.2:
            tier = "excellent"
        elif pue < 1.4:
            tier = "good"
        elif pue < 1.6:
            tier = "average"
        else:
            tier = "poor"

        return PluginResult(
            name=self.name,
            ok=True,
            data={
                "pue": pue,
                "overhead_mw": overhead_mw,
                "efficiency_pct": round(efficiency_pct, 2),
                "tier": tier,
            },
            metadata={},
        )

    def teardown(self, context: PluginContext) -> None:
        pass

## Step 2 — Standalone Testing

Before registering a plugin with the engine, it is good practice to test it in
isolation. This lets you validate the core logic without worrying about engine
configuration, payload schemas, or interactions with other plugins.

We create a minimal `PluginContext` with an empty config, instantiate the plugin,
call `initialize`, and then pass a representative payload to `execute`.

In [ ]:
# Build a lightweight context for standalone testing.
test_context = PluginContext(config={})

plugin = PUEEfficiencyPlugin()
plugin.initialize(test_context)

sample_payload = {
    "pue": 1.35,
    "power_mw": 12.0,
    "it_load_mw": 8.9,
}

result = plugin.execute(sample_payload, test_context)

print(f"Plugin : {result.name}")
print(f"OK     : {result.ok}")
print(f"Tier   : {result.data['tier']}")
print(f"Overhead MW   : {result.data['overhead_mw']}")
print(f"Efficiency %  : {result.data['efficiency_pct']}")

## Step 3 — Register with the Engine

Once the plugin passes standalone tests, we register it with `EWISEngine`. The
engine's `PluginManager.register(plugin)` call invokes `plugin.initialize(context)`
automatically, so you do not need to call it yourself. When the engine runs,
`PluginManager.run_all(payload)` calls `execute` on every registered plugin and
collects their `PluginResult` objects into the final report.

In [ ]:
from ewis.core.engine import EWISEngine

engine = EWISEngine()
engine.register(PUEEfficiencyPlugin())

payload = {
    "datacenter_id": "DC-TRAIN-01",
    "region": "TEST",
    "timestamp_utc": "2026-03-03T12:00:00Z",
    "pue": 1.18,
    "power_mw": 25.0,
    "it_load_mw": 21.2,
    "ambient_temp_c": 22.5,
}

report = engine.run(payload)

print(report.summary())
print("---")

# Retrieve the PUE plugin result from the report.
pue_result = next(r for r in report.plugin_results if r.name == "pue_efficiency")
print(f"PUE tier: {pue_result.data['tier']}")
print(f"Efficiency: {pue_result.data['efficiency_pct']}%")

## Step 4 — Multiple Plugins Together

The real power of the plugin system comes from composition. You can register
any number of plugins on the same engine. Each receives the same payload and
returns its own `PluginResult`, so plugins remain decoupled from one another
while contributing to a unified report.

In [ ]:
from ewis.plugins.builtin.cooling_optimizer import CoolingOptimizerPlugin

engine = EWISEngine()
engine.register(PUEEfficiencyPlugin())
engine.register(CoolingOptimizerPlugin())

payload = {
    "datacenter_id": "DC-TRAIN-02",
    "region": "TEST",
    "timestamp_utc": "2026-03-03T12:00:00Z",
    "pue": 1.52,
    "power_mw": 30.0,
    "it_load_mw": 19.7,
    "ambient_temp_c": 31.0,
}

report = engine.run(payload)

for pr in report.plugin_results:
    status = "PASS" if pr.ok else "FAIL"
    print(f"[{status}] {pr.name}: {pr.data}")

## Step 5 — Using the Plugin Context

The `PluginContext` object provides a `cache` dictionary that persists across
successive engine runs within the same session. This is useful when a plugin
needs to track cumulative state — for example, counting invocations, computing
running averages, or storing expensive lookup data that should only be loaded
once.

You can also read engine-level configuration through `context.config` or the
safe `context.get(key, default)` accessor.

In [ ]:
class StatefulCounterPlugin(BasePlugin):
    """Tracks how many times the plugin has been invoked using context.cache."""

    def __init__(self, name: str = "invocation_counter") -> None:
        super().__init__(name)

    def initialize(self, context: PluginContext) -> None:
        # Seed the counter in the shared cache.
        context.cache.setdefault("invocation_count", 0)

    def execute(self, payload: dict, context: PluginContext) -> PluginResult:
        context.cache["invocation_count"] += 1
        count = context.cache["invocation_count"]

        return PluginResult(
            name=self.name,
            ok=True,
            data={"invocation_count": count},
            metadata={},
        )

    def teardown(self, context: PluginContext) -> None:
        pass


# Demonstrate cache persistence across two successive engine runs.
ctx = PluginContext(config={})
counter = StatefulCounterPlugin()
counter.initialize(ctx)

for i in range(1, 3):
    result = counter.execute({}, ctx)
    print(f"Run {i} -> invocation_count = {result.data['invocation_count']}")

## Step 6 — Error Handling in Plugins

A well-behaved plugin should **never** raise unhandled exceptions. Instead,
catch anticipated errors and return a `PluginResult` with `ok=False` and a
descriptive error message in the `data` dictionary. This lets the engine
continue running the remaining plugins and produce a complete report, even
when one plugin encounters bad input.

In [ ]:
class SafeDivisionPlugin(BasePlugin):
    """Divides power_mw by a config-supplied divisor, handling errors gracefully."""

    def __init__(self, name: str = "safe_division") -> None:
        super().__init__(name)

    def initialize(self, context: PluginContext) -> None:
        pass

    def execute(self, payload: dict, context: PluginContext) -> PluginResult:
        divisor = context.get("divisor", None)

        if divisor is None or divisor == 0:
            return PluginResult(
                name=self.name,
                ok=False,
                data={"error": "divisor is missing or zero"},
                metadata={},
            )

        power_mw = payload.get("power_mw", 0.0)
        quotient = power_mw / divisor

        return PluginResult(
            name=self.name,
            ok=True,
            data={"quotient": round(quotient, 4)},
            metadata={},
        )

    def teardown(self, context: PluginContext) -> None:
        pass


# --- Success case: valid divisor ---
ctx_ok = PluginContext(config={"divisor": 4})
div_plugin = SafeDivisionPlugin()
div_plugin.initialize(ctx_ok)

result_ok = div_plugin.execute({"power_mw": 20.0}, ctx_ok)
print(f"Success -> ok={result_ok.ok}, data={result_ok.data}")

# --- Failure case: divisor is zero ---
ctx_bad = PluginContext(config={"divisor": 0})
div_plugin_bad = SafeDivisionPlugin()
div_plugin_bad.initialize(ctx_bad)

result_bad = div_plugin_bad.execute({"power_mw": 20.0}, ctx_bad)
print(f"Failure -> ok={result_bad.ok}, data={result_bad.data}")

## Plugin Development Checklist

Before shipping your plugin, make sure you can tick every box:

- [ ] Inherit from `BasePlugin`
- [ ] Implement `initialize()` and `execute()` (both required)
- [ ] Return a `PluginResult` with descriptive name, ok status, and data
- [ ] Keep plugins deterministic — avoid network calls without caching
- [ ] Use `context.cache` for state that should persist across calls
- [ ] Handle missing payload fields gracefully
- [ ] Write unit tests for your plugin independently of the engine